# Best-fit $\chi^2$ summary

Read the global best-fit $\chi^2$ and active-bin count reported by PROfit for every NuWro, open-data, and Asimov fit, and split the minimum into its three parts, as in `21_prefit_postfit_distributions.ipynb`:

$$\chi^2_{\rm tot} = \underbrace{r^T C^{-1} r}_{\rm stat} + \underbrace{\hat\alpha^T\hat\alpha}_{\rm syst} + \underbrace{\hat\theta^T\hat\theta}_{\rm pull},$$

with $r$ the residual of the constrained best fit (`bestfit2d`) against the Neyman data statistics $C={\rm diag}(d)$ (zero-data bins excluded), $\hat\alpha$ the posterior pulls of the covariance-type systematic modes (`covmode_pulls`) and $\hat\theta$ the fitted parameters. Uniform parameters contribute zero pull penalty; other Gaussian-constrained nuisance parameters in the same fit still contribute. $\chi^2_{\rm data}=\chi^2_{\rm tot}-\chi^2_{\rm pull}$ comes from the log; stat + syst reproduces it exactly (Woodbury), and the mismatch is kept as a check column. The number of degrees of freedom is $\nu=N_{\rm bins}-N_{\rm flat}$, where $N_{\rm flat}$ counts the fitted parameters with a flat prior: a Gaussian-constrained parameter adds one parameter and one prior pseudo-measurement, which cancel. Missing or unfinished fits, and `PROfile.root` files without `covmode_pulls`, are retained as `NaN` so gaps are visible.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import uproot
from IPython.display import display

OUTPUT_ROOT = Path("/nevis/hopper/data/epelaez/axial_mass")
CHANNEL = "nu_uBooNE_numuCC1p"

SUITES = {
    "NuWro": "nuwro_fit_results",
    "Open data": "opendata_fit_results",
    "Asimov": "asimov_fit_results",
}
FIT_ORDER = [
    "ma", "ma_uniform", "minerva_k8", "minerva_k7",
    "minerva_k6", "lqcd_k7", "lqcd_k6", "minerva_lqcd_k7", "minerva_lqcd_k6",
    "minerva_k6_nuisance", "lqcd_k6_nuisance", "minerva_lqcd_k6_nuisance",
    "minerva_k6_uniform", "minerva_k6_uniform_nuisance",
]
# Fitted parameters: the axial ones plus, in the M_A and *_nuisance fits, the five floated
# modeling knobs (RPA_CCQE, XSecShape_CCMEC, MFP_N, DecayAngMEC, FrCEx_N; since 2026-10-03).
N_FITTED = {
    "ma": 6, "ma_uniform": 6,
    "lqcd_k6": 2, "lqcd_k7": 3, "minerva_k6": 2, "minerva_k7": 3,
    "minerva_k8": 4, "minerva_lqcd_k6": 2, "minerva_lqcd_k7": 3,
    "minerva_k6_uniform": 2, "minerva_k6_nuisance": 7, "lqcd_k6_nuisance": 7, "minerva_lqcd_k6_nuisance": 7,
    "minerva_k6_uniform_nuisance": 7,
}
UNIFORM_PARAMETERS = {
    "ma_uniform": {"MACCQE"},
    "minerva_k6_uniform": {"FAzexpMinervaK6PCA1", "FAzexpMinervaK6PCA2"},
    "minerva_k6_uniform_nuisance": {"FAzexpMinervaK6PCA1", "FAzexpMinervaK6PCA2"},
}
CHI2_PATTERN = re.compile(
    r"Global Best Fit chi(?:\^?2|²):\s*"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
)
NBINS_PATTERN = re.compile(r"/\s*(\d+)\s+reco bins")
# The emitting function changed between PROfit builds (main -> print_global_fit_results),
# so match on "|| <name> : <value>" instead of the function name. Requiring whitespace
# before the colon keeps the trailing "Total run time: 63 seconds" line out.
PARAM_PATTERN = re.compile(
    r"\|\|\s+(\w+)\s+:\s*"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
)

In [ ]:
def read_fit_stats(log_file):
    """Return total chi2, active-bin count, and best-fit parameters."""
    if not log_file.is_file():
        return float("nan"), float("nan"), {}
    text = log_file.read_text(errors="replace")
    chi2_matches = CHI2_PATTERN.findall(text)
    nbins_matches = NBINS_PATTERN.findall(text)
    chi2 = float(chi2_matches[-1]) if chi2_matches else float("nan")
    nbins = int(nbins_matches[-1]) if nbins_matches else float("nan")
    final_block = text.rsplit("Global Best Fit chi^2:", 1)[-1]
    parameters = {name: float(value) for name, value in PARAM_PATTERN.findall(final_block)}
    return chi2, nbins, parameters

def pull_penalty(fit, parameters):
    """Calculate the Gaussian pull term; uniform parameters contribute zero."""
    uniform = UNIFORM_PARAMETERS.get(fit, set())
    penalty = 0.0
    for name, value in parameters.items():
        if name in uniform:
            continue
        penalty += value ** 2
    return penalty

def stat_syst_split(profile_path):
    """(stat, syst) parts of PROfit's data term, as in 21_prefit_postfit_distributions.ipynb.
    stat: Neyman residual of the constrained best fit, zero-data bins excluded; syst: squared
    posterior pulls of the covariance-type systematic modes. NaN when either is unavailable."""
    if not profile_path.is_file():
        return float("nan"), float("nan")
    with uproot.open(profile_path) as profile_file:
        if "covmode_pulls" not in profile_file:
            return float("nan"), float("nan")
        data = profile_file[f"{CHANNEL}_data2d"].values().reshape(-1)
        bestfit = profile_file[f"{CHANNEL}_bestfit2d"].values().reshape(-1)
        syst = float(np.sum(profile_file["covmode_pulls"].values(axis="y") ** 2))
    use = data > 0
    stat = float(np.sum((data[use] - bestfit[use]) ** 2 / data[use]))
    return stat, syst

records = []
for sample, suite in SUITES.items():
    for fit in FIT_ORDER:
        log_file = OUTPUT_ROOT / suite / fit / "profile.log"
        chi2_tot, nbins, parameters = read_fit_stats(log_file)
        if log_file.is_file() and not parameters:
            raise RuntimeError(
                f"{log_file}: found no best-fit parameters. The penalty term would "
                "silently come out zero and chi2_data would equal chi2_tot; check "
                "PARAM_PATTERN against the log format."
            )
        chi2_penalty = pull_penalty(fit, parameters)
        chi2_data = chi2_tot - chi2_penalty
        chi2_stat, chi2_syst = stat_syst_split(log_file.parent / f"{fit}_v1_PROfile.root")
        n_fitted = N_FITTED[fit]
        # Gaussian-constrained parameters add a prior pseudo-measurement that cancels their
        # degree of freedom, so only flat-prior parameters reduce nu.
        n_flat = len(UNIFORM_PARAMETERS.get(fit, ()))
        ndof = nbins - n_flat
        records.append({
            "fit": fit,
            "sample": sample,
            "chi2_stat": chi2_stat,
            "chi2_syst": chi2_syst,
            "chi2_penalty": chi2_penalty,
            "chi2_data": chi2_data,
            "chi2_tot": chi2_tot,
            "split_mismatch": chi2_stat + chi2_syst - chi2_data,
            "nbins": nbins,
            "n_fitted": n_fitted,
            "n_flat": n_flat,
            "ndof": ndof,
            "chi2_tot_per_ndof": chi2_tot / ndof,
            "profile_log": str(log_file),
        })

chi2_long = pd.DataFrame(records)
large_mismatch = chi2_long[chi2_long.split_mismatch.abs() > 0.5]
if not large_mismatch.empty:
    print("warning: stat + syst differs from the log's data term by more than 0.5 for")
    display(large_mismatch[["fit", "sample", "chi2_data", "split_mismatch"]])

In [ ]:
shared_columns = ["nbins", "n_fitted", "n_flat", "ndof"]
shared = chi2_long.groupby("fit")[shared_columns].first().reindex(FIT_ORDER)
shared.columns = pd.MultiIndex.from_product(
    [shared_columns, [""]], names=["quantity", "Data sample"]
)
sample_columns = [
    "chi2_stat", "chi2_syst", "chi2_penalty", "chi2_data", "chi2_tot", "chi2_tot_per_ndof",
    "split_mismatch",
]
sample_results = chi2_long.pivot(
    index="fit", columns="sample", values=sample_columns,
).reindex(index=FIT_ORDER)
sample_results = sample_results.reindex(
    columns=pd.MultiIndex.from_product(
        [sample_columns, list(SUITES)],
        names=["quantity", "Data sample"],
    )
)
chi2_table = pd.concat([shared, sample_results], axis=1)
chi2_table.index.name = "Fit"
display(chi2_table.style.format({
    **{column: "{:.4f}" for column in sample_columns},
    "split_mismatch": "{:.2e}",
    "nbins": "{:.0f}",
    "n_fitted": "{:.0f}",
    "n_flat": "{:.0f}",
    "ndof": "{:.0f}",
}, na_rep="not available"))

The long-form table below includes the source log path, which is useful for checking any missing entry.

In [ ]:
display(chi2_long)

In [ ]:
LATEX_FIT_ORDER = [
    "ma", "ma_uniform", "minerva_k8", "minerva_k7",
    "minerva_k6", "lqcd_k7", "lqcd_k6", "minerva_lqcd_k7", "minerva_lqcd_k6",
    "minerva_k6_nuisance", "lqcd_k6_nuisance", "minerva_lqcd_k6_nuisance",
    "minerva_k6_uniform", "minerva_k6_uniform_nuisance",
]
LATEX_FIT_LABELS = {
    "ma": r"$M_A$ Gaussian prior",
    "ma_uniform": r"$M_A$ uniform prior",
    "minerva_k8": r"MINERvA, $k_{\max}=8$",
    "minerva_k7": r"MINERvA, $k_{\max}=7$",
    "minerva_k6": r"MINERvA, $k_{\max}=6$",
    "lqcd_k6": r"LQCD, $k_{\max}=6$",
    "lqcd_k7": r"LQCD, $k_{\max}=7$",
    "minerva_lqcd_k7": r"MINERvA+LQCD, $k_{\max}=7$",
    "minerva_lqcd_k6": r"MINERvA+LQCD, $k_{\max}=6$",
    "minerva_k6_nuisance": r"MINERvA with nuisance, $k_{\max}=6$",
    "lqcd_k6_nuisance": r"LQCD with nuisance, $k_{\max}=6$",
    "minerva_lqcd_k6_nuisance": r"MINERvA+LQCD with nuisance, $k_{\max}=6$",
    "minerva_k6_uniform": r"Uniform prior, $k_{\max}=6$",
    "minerva_k6_uniform_nuisance": r"Uniform with nuisance, $k_{\max}=6$",
}

oepndata_results = chi2_long.query("sample == 'Open data'").set_index("fit")
latex_lines = [
    r"\begin{ruledtabular}",
    r"    \begin{tabular}{lcccccc}",
    r"        Fit & $\chi^2_{\mathrm{stat}}$ & $\chi^2_{\mathrm{syst}}$ & $\chi^2_{\mathrm{pull}}$ & $\chi^2_{\mathrm{tot}}$ & $\nu$ & $\chi^2_{\mathrm{tot}}/\nu$ \\",
    r"        \hline",
]
for fit in LATEX_FIT_ORDER:
    row = oepndata_results.loc[fit]
    # A dash marks fits with no Gaussian-constrained parameter, whose pull term is zero by
    # construction rather than small.
    no_gaussian = len(UNIFORM_PARAMETERS.get(fit, ())) == N_FITTED[fit]
    pull = "--" if no_gaussian else f"{row.chi2_penalty:.2f}"
    latex_lines.append(
        f"        {LATEX_FIT_LABELS[fit]} & {row.chi2_stat:.2f} & "
        f"{row.chi2_syst:.2f} & {pull} & "
        f"{row.chi2_tot:.2f} & {row.ndof:.0f} & {row.chi2_tot_per_ndof:.2f} \\\\"
    )
latex_lines.extend([r"    \end{tabular}", r"\end{ruledtabular}"])
print("\n".join(latex_lines))